Robot Localization with Python and Particle Filters
===================================================

Import libraries and load map.

In [57]:
import numpy as np
import cv2

map = cv2.imread("map.png", cv2.IMREAD_GRAYSCALE)
HEIGHT, WIDTH = map.shape
print(map)

rx, ry, rtheta = WIDTH / 4., HEIGHT / 4., 0

[[ 79  80  82 ... 133 148 156]
 [ 78  79  81 ... 138 156 164]
 [ 76  77  79 ... 147 170 180]
 ...
 [181 181 182 ... 174 172 171]
 [178 178 178 ... 180 179 178]
 [177 177 177 ... 183 182 182]]


Map coordinate system

![title](images/coords.png)

CAUTION: The terrain height at X,Y coordinates is map(Y,X).

Read keyboard input.

In [58]:
STEP_PIXELS = 5 # 5 pixels
TURN_RADIANS = np.radians(25) # one press is equal to 25 degrees

FWD_KEY = 82
RIGHT_KEY = 83
LEFT_KEY = 81

def get_input():
    fwd, turn, halt = 0, 0, False
    k = cv2.waitKey()
    if k == FWD_KEY:
        fwd = STEP_PIXELS
    elif k == RIGHT_KEY:
        turn = TURN_RADIANS
    elif k == LEFT_KEY:
        turn = -TURN_RADIANS
    else:
        halt = True

    return fwd, turn, halt

Move the robot, with Gausssian noise.

![title](images/gaussian.png)

In [59]:
SIGMA_FWD = 0.5
SIGMA_TURN = np.radians(5)

def move_robot(rx, ry, rtheta, fwd, turn):
    fwd_noisy = np.random.normal(fwd, SIGMA_FWD, 1)
    rx += fwd_noisy * np.cos(rtheta)
    ry += fwd_noisy * np.sin(rtheta)
    print(f"fwd_noisy {fwd_noisy}")
    
    turn_noisy = np.random.normal(rtheta, SIGMA_TURN, 1)
    rtheta += turn_noisy
    rtheta = np.clip(rtheta, 0, 2*np.pi)
    print(f"turn_noisy {turn_noisy}")
    return rx, ry, rtheta

Initialize particle cloud.

In [60]:
NUM_PARTICLES = 3000


def init():
    particles = np.random.rand(NUM_PARTICLES, 3)
    particles *= np.array([WIDTH, HEIGHT, np.radians(360)])
    return particles

Move the particles.

In [61]:
def move_particles(particles, fwd, turn):
    particles[:, 0] += fwd * np.cos(turn)
    particles[:, 1] += fwd * np.sin(turn)
    particles[:, 2] += turn
    particles[:, 0] = np.clip(particles[:, 0], 0, WIDTH-1)
    particles[:, 1] = np.clip(particles[:, 1], 0, HEIGHT-1)
    return particles

Get value from robot's sensor.

In [62]:
SIGMA_SENSOR = 2
def sense(x, y, noisy=False):
    x = int(x)
    y = int(y)
    val = map[y, x]
    if noisy:
        return np.random.normal(val, SIGMA_SENSOR, 1)
    
    return val

Compute particle weights.

In [63]:
def compute_weights(particles, robot_sensor):    
    errors = np.zeros(NUM_PARTICLES)
    for i in range(NUM_PARTICLES):
        elevation = sense(particles[i, 0], particles[i, 1], noisy=False)
        errors[i] = abs(robot_sensor - elevation)

    weights = np.max(errors) - errors
    weights[
        (particles[:, 0] == 0) |  (particles[:, 0] == WIDTH - 1) |
        (particles[:, 1] == 0) |  (particles[:, 1] == HEIGHT - 1)
    ] = 0.
    
    return weights ** 3

Resample the particles.

In [64]:
def resample(particles, weights):
    probabilities = weights / np.sum(weights)
    new_index = np.random.choice(NUM_PARTICLES, size = NUM_PARTICLES, p = probabilities)
    particles = particles[new_index, :]
    return particles

Add noise to the particles.

In [65]:
SIGMA_POSITION = 2
SIGMA_TURN = np.radians(10)

def add_noise(particles):
    noise = np.concatenate(
        [np.random.normal(0, SIGMA_POSITION, (NUM_PARTICLES, 1)),
         np.random.normal(0, SIGMA_POSITION, (NUM_PARTICLES, 1)),
         np.random.normal(0, SIGMA_TURN, (NUM_PARTICLES, 1)),
        ],
        axis = -1
    )
    particles += noise
    return particles

Display robot, particles and best guess.

In [66]:
def display(map, rx, ry, particles):
    lmap = cv2.cvtColor(map, cv2.COLOR_GRAY2BGR)
    print("****")
    
    # Display particles
    if len(particles) > 0:
        for i in range(NUM_PARTICLES):
            cv2.circle(lmap, 
                       (int(particles[i,0]), int(particles[i,1])), 
                       1, 
                       (255,0,0), 
                       1)
        
    # Display robot
    cv2.circle(lmap, (int(rx), int(ry)), 5, (0,255,0), 10)

    # Display best guess
    if len(particles) > 0:
        px = np.mean(particles[:,0])
        py = np.mean(particles[:,1])
        cv2.circle(lmap, (int(px), int(py)), 5, (0,0,255), 5)

    cv2.imshow('map', lmap)

Main routine.

In [67]:
particles = init()
while True:
    display(map, rx, ry, particles)
    fwd, turn, halt = get_input()
    if halt:
        break
    rx, ry, rtheta = move_robot(rx, ry, rtheta, fwd, turn)
    particles = move_particles(particles, fwd, turn)
    if fwd != 0:
        robot_sensor = sense(rx, ry, noisy=True)        
        weights = compute_weights(particles, robot_sensor)
        particles = resample(particles, weights)
        particles = add_noise(particles)
    
cv2.destroyAllWindows()                        


****


QObject::moveToThread: Current thread (0x2ce96040) is not the object's thread (0x2d0bfea0).
Cannot move to target thread (0x2ce96040)

QObject::moveToThread: Current thread (0x2ce96040) is not the object's thread (0x2d0bfea0).
Cannot move to target thread (0x2ce96040)

QObject::moveToThread: Current thread (0x2ce96040) is not the object's thread (0x2d0bfea0).
Cannot move to target thread (0x2ce96040)

QObject::moveToThread: Current thread (0x2ce96040) is not the object's thread (0x2d0bfea0).
Cannot move to target thread (0x2ce96040)

QObject::moveToThread: Current thread (0x2ce96040) is not the object's thread (0x2d0bfea0).
Cannot move to target thread (0x2ce96040)

QObject::moveToThread: Current thread (0x2ce96040) is not the object's thread (0x2d0bfea0).
Cannot move to target thread (0x2ce96040)

QObject::moveToThread: Current thread (0x2ce96040) is not the object's thread (0x2d0bfea0).
Cannot move to target thread (0x2ce96040)

QObject::moveToThread: Current thread (0x2ce96040) is n

fwd_noisy [4.65341566]
turn_noisy [-0.036928]
****


/tmp/ipykernel_11674/3863653360.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  x = int(x)
/tmp/ipykernel_11674/3863653360.py:4: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  y = int(y)
/tmp/ipykernel_11674/2288787860.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  errors[i] = abs(robot_sensor - elevation)
/tmp/ipykernel_11674/3756462798.py:15: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensu

fwd_noisy [4.75392484]
turn_noisy [0.21194956]
****
fwd_noisy [5.23444713]
turn_noisy [0.03611955]
****
fwd_noisy [4.90461478]
turn_noisy [0.61326847]
****
fwd_noisy [-0.48022527]
turn_noisy [1.00360759]
****
fwd_noisy [0.922296]
turn_noisy [2.0070487]
****
fwd_noisy [-0.02904856]
turn_noisy [3.82461537]
****
fwd_noisy [0.27247705]
turn_noisy [6.04157059]
****
fwd_noisy [-0.02570574]
turn_noisy [6.30147603]
****
fwd_noisy [-0.42922433]
turn_noisy [6.42081276]
****
fwd_noisy [-0.69475231]
turn_noisy [6.50335923]
****
fwd_noisy [0.15309012]
turn_noisy [6.19760604]
****
fwd_noisy [5.87736643]
turn_noisy [6.10897768]
****
fwd_noisy [5.34847591]
turn_noisy [6.4904015]
****
fwd_noisy [5.48518443]
turn_noisy [6.0162696]
****
fwd_noisy [5.46438484]
turn_noisy [6.42174809]
****
fwd_noisy [5.68931051]
turn_noisy [6.43175578]
****
fwd_noisy [5.14233748]
turn_noisy [6.1308572]
****
fwd_noisy [5.79218761]
turn_noisy [6.40122398]
****
fwd_noisy [4.87654623]
turn_noisy [6.53351375]
****
fwd_noisy [0.

IndexError: index 303 is out of bounds for axis 1 with size 300